**A09 BAGGING**

In [2]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("Default.csv")
target_col = "default"
X = df.drop(columns=[target_col])
y = df[target_col]

In [5]:
#CONVERTIR y A BINARIA
if pd.api.types.is_numeric_dtype(y):
    y = (y > 0).astype(int)
else:
    y = y.astype(str).str.lower()
    positive_vals = ["yes", "true", "1", "default", "si", "sí"]
    y = y.apply(lambda v: 1 if v in positive_vals else 0).astype(int)

y.unique()

array([0, 1])

In [6]:
#CONVERTIR X A NUMÉRICA
X = pd.get_dummies(X, drop_first=True)

In [7]:
#BOOTSTRAP
N_BOOT = 5000
Ypred = np.zeros((len(y), N_BOOT))

In [8]:
#BOOTSTRAP + ÁRBOLES
for b in range(N_BOOT):

    #Seleccionar 2 columnas RANDOM ya numéricas
    cols = np.random.choice(X.columns, 2, replace=False)
    X_sub = X[cols]

    #Bootstrap
    idx = np.random.choice(len(X), len(X), replace=True)
    X_boot = X_sub.iloc[idx]
    y_boot = y.iloc[idx]

    #Árbol
    clf = DecisionTreeClassifier(max_depth=4)
    clf.fit(X_boot, y_boot)

    #Guardar predicciones
    Ypred[:, b] = clf.predict(X_sub)

In [9]:
#ENSAMBLE
mean_pred = Ypred.mean(axis=1)

In [10]:
#UMBRAL
threshold = 0.05
final_pred = (mean_pred >= threshold).astype(int)

In [11]:
threshold

0.05

In [12]:
#ACCURACY
acc = accuracy_score(y, final_pred)
acc

0.9737

**Conclusión:**
El modelo funcionó bien: después de hacer 5000 bootstraps y muchos árboles, logró predecir correctamente el 97% de los casos. Eso significa que combinar muchos modelos pequeños ayuda a tomar una mejor decisión que un solo árbol. Además, el accuracy no fue perfecto (no dio 1), lo cual es bueno porque quiere decir que el modelo no memorizó los datos, sino que realmente aprendió patrones.
- En resumen, el ensamble de árboles fue estable, acertado y cumplió lo que se pedía.